# Water Potability Classification

**Classification Project 18 of 100** - Predict whether water is safe to drink from chemical properties.

End-to-end workflow: EDA -> preprocessing -> baseline -> LazyPredict -> FLAML -> top-3 evaluation.

## 2. Project Overview

Access to clean drinking water is essential for health. This notebook builds a binary classifier predicting potability (Potability=1) from 9 water chemistry parameters. This is a challenging dataset: features overlap significantly between classes, and ~14% of values are missing.

## 3. Learning Objectives

1. Work with a hard classification problem (classes overlap heavily)
2. Handle significant missing values (~14-24%)
3. Build proper imputation pipelines
4. Understand fundamental accuracy limits
5. Practice with all-numeric features (9 measurements)
6. Use appropriate metrics (~39% potable)
7. Benchmark with LazyPredict and FLAML
8. Accept realistic limits - not every dataset yields 95% accuracy

## 4. Problem Statement

> Given 9 water quality parameters (pH, hardness, solids, chloramines, sulfate, conductivity, organic carbon, trihalomethanes, turbidity), predict whether the water is potable.

Binary classification. This is a hard problem - expect F1 in the 0.55-0.70 range.

## 5. Why This Project Matters

| Stakeholder | Impact |
|---|---|
| Water utilities | Automate potability screening |
| Public health | Early contamination detection |
| WHO/regulators | Data-driven quality thresholds |
| ML learners | Not all problems are easy - accept realistic limits |

## 6. Dataset Overview

| Property | Value |
|---|---|
| Kaggle slug | adityakadiwal/water-potability |
| Rows | ~3,276 |
| Features | 9 numeric |
| Target | Potability (0=not potable, 1=potable) |
| Class balance | ~61% not potable, ~39% potable |
| Missing | ~14% pH, ~24% Sulfate, ~5% Trihalomethanes |

## 7. Dataset Source and License Notes

- Kaggle: https://www.kaggle.com/datasets/adityakadiwal/water-potability
- License: CC0 Public Domain.
- Note: Synthetic dataset inspired by WHO guidelines.

## 8. Environment Setup

In [ ]:
import subprocess, sys, importlib
for pkg in ["lazypredict","flaml","kagglehub","xgboost","lightgbm"]:
    try: importlib.import_module(pkg)
    except ImportError: subprocess.check_call([sys.executable,"-m","pip","install","-q",pkg])
print("Environment ready.")

## 9. Imports

In [ ]:
import os, warnings, glob
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay, classification_report)
from lazypredict.Supervised import LazyClassifier
from flaml import AutoML
warnings.filterwarnings("ignore"); sns.set_theme(style="whitegrid"); SEED=42
print("All imports loaded.")

## 10. Configuration / Constants

In [ ]:
KAGGLE_SLUG = "adityakadiwal/water-potability"
TARGET = "Potability"
TEST_SIZE = 0.15; VAL_SIZE = 0.15; SEED = 42; FLAML_BUDGET = 120
print(f"Target: {TARGET}")

## 11. Dataset Download or Loading

In [ ]:
import kagglehub
try:
    path = kagglehub.dataset_download(KAGGLE_SLUG)
    print(f"Downloaded to: {path}")
except Exception as e:
    raise RuntimeError(f"Download failed: {e}")
csv_files = glob.glob(os.path.join(path,"**","*.csv"), recursive=True)
df_raw = pd.read_csv(csv_files[0])
print(f"Shape: {df_raw.shape}")
df_raw.head()

## 12. Data Validation Checks

In [ ]:
print("Missing values:")
print(df_raw.isnull().sum())
df = df_raw.copy()
dupes = df.duplicated().sum()
print(f"Duplicates: {dupes}")
if dupes > 0: df = df.drop_duplicates().reset_index(drop=True)
assert TARGET in df.columns
print(f"\nTarget:\n{df[TARGET].value_counts()}")
print(f"Potable rate: {df[TARGET].mean():.3f}")

## 13. Exploratory Data Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(5,4))
df[TARGET].value_counts().plot.bar(ax=ax, color=["steelblue","salmon"])
ax.set_title("Potability Distribution"); ax.set_xticklabels(["Not Potable","Potable"], rotation=0)
plt.tight_layout(); plt.show()

In [ ]:
feat_cols = [c for c in df.columns if c != TARGET]
n = len(feat_cols); rows = (n+2)//3
fig, axes = plt.subplots(rows, 3, figsize=(16, 4*rows))
for ax, col in zip(axes.ravel(), feat_cols):
    for lb in [0,1]:
        ax.hist(df[df[TARGET]==lb][col].dropna(), bins=30, alpha=0.5, label="Potable" if lb else "Not Potable")
    ax.set_title(col); ax.legend(fontsize=7)
for ax in axes.ravel()[n:]: ax.set_visible(False)
plt.tight_layout(); plt.show()

In [ ]:
corr = df.corr()[TARGET].drop(TARGET).sort_values(key=abs, ascending=False)
print("Correlations with target:")
print(corr)
print("\nNote: Very weak correlations - fundamentally hard problem.")

## 14. Target Analysis

~39% potable. Extremely weak correlations explain why this is hard.

In [ ]:
print(f"Potable rate: {df[TARGET].mean():.1%}")
print(f"Majority baseline: {max(df[TARGET].mean(), 1-df[TARGET].mean()):.1%}")

## 15. Train / Validation / Test Split

In [ ]:
X = df.drop(columns=[TARGET]); y = df[TARGET]
X_tmp, X_test, y_tmp, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_tmp, y_tmp, test_size=VAL_SIZE/(1-TEST_SIZE), random_state=SEED, stratify=y_tmp)
print(f"Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}")

## 16. Preprocessing Strategy

All numeric. Median impute + StandardScaler.

In [ ]:
num_cols = X_train.columns.tolist()
preprocessor = ColumnTransformer(transformers=[
    ("num", Pipeline([("imp",SimpleImputer(strategy="median")),("sc",StandardScaler())]), num_cols)
], remainder="passthrough")
print(f"Features: {len(num_cols)}")

## 17. Feature Engineering

- PH_DEVIATION: absolute deviation from pH 7
- HARDNESS_SOLIDS_RATIO
- CHLOR_TURBIDITY interaction

In [ ]:
def eng(d):
    d = d.copy()
    ph_cols = [c for c in d.columns if c.lower()=="ph"]
    if ph_cols: d["PH_DEVIATION"] = (d[ph_cols[0]] - 7).abs()
    h = [c for c in d.columns if "hardness" in c.lower()]
    s = [c for c in d.columns if "solids" in c.lower()]
    if h and s: d["HARDNESS_SOLIDS_RATIO"] = d[h[0]] / d[s[0]].clip(lower=1)
    ch = [c for c in d.columns if "chlor" in c.lower()]
    t = [c for c in d.columns if "turb" in c.lower()]
    if ch and t: d["CHLOR_TURBIDITY"] = d[ch[0]] * d[t[0]]
    return d
X_train=eng(X_train); X_val=eng(X_val); X_test=eng(X_test)
num_cols = X_train.columns.tolist()
preprocessor = ColumnTransformer(transformers=[
    ("num", Pipeline([("imp",SimpleImputer(strategy="median")),("sc",StandardScaler())]), num_cols)
], remainder="passthrough")
print(f"Features: {X_train.shape[1]}")

## 18. Baseline Model

In [ ]:
results = {}
for name, clf in [("Dummy",DummyClassifier(strategy="most_frequent",random_state=SEED)),
                  ("LogReg",LogisticRegression(max_iter=1000,class_weight="balanced",random_state=SEED)),
                  ("RF",RandomForestClassifier(n_estimators=200,class_weight="balanced",random_state=SEED,n_jobs=-1))]:
    pipe = Pipeline([("pre",preprocessor),("clf",clf)])
    pipe.fit(X_train, y_train); yp=pipe.predict(X_val)
    r={"Accuracy":accuracy_score(y_val,yp),"F1":f1_score(y_val,yp),"Recall":recall_score(y_val,yp)}
    if hasattr(clf,"predict_proba"): r["ROC-AUC"]=roc_auc_score(y_val,pipe.predict_proba(X_val)[:,1])
    results[name]=r; print(f"{name}: {r}")

## 19. LazyPredict Benchmark

In [ ]:
X_train_lp=preprocessor.fit_transform(X_train); X_val_lp=preprocessor.transform(X_val)
lazy=LazyClassifier(verbose=0,ignore_warnings=True,random_state=SEED)
models_lp,_=lazy.fit(X_train_lp,X_val_lp,y_train,y_val)
models_lp.head(15)

## 20. FLAML AutoML Run

In [ ]:
automl=AutoML()
automl.fit(X_train, y_train, task="classification", metric="f1", time_budget=FLAML_BUDGET, seed=SEED, verbose=0)
print(f"Best: {automl.best_estimator}")
yp_fl=automl.predict(X_val); ypr_fl=automl.predict_proba(X_val)[:,1]
results["FLAML"]={"Accuracy":accuracy_score(y_val,yp_fl),"F1":f1_score(y_val,yp_fl),"ROC-AUC":roc_auc_score(y_val,ypr_fl)}
print(results["FLAML"])

## 21. Top 3 Model Selection

In [ ]:
try:
    from lightgbm import LGBMClassifier; has_lgbm=True
except ImportError: has_lgbm=False
try:
    from xgboost import XGBClassifier; has_xgb=True
except ImportError: has_xgb=False
w=(y_train==0).sum()/max((y_train==1).sum(),1)
top3={}
if has_lgbm: top3["LightGBM"]=LGBMClassifier(n_estimators=500,learning_rate=0.05,max_depth=5,class_weight="balanced",random_state=SEED,verbose=-1,n_jobs=-1)
else:
    from sklearn.ensemble import ExtraTreesClassifier
    top3["ExtraTrees"]=ExtraTreesClassifier(n_estimators=500,class_weight="balanced",random_state=SEED,n_jobs=-1)
if has_xgb: top3["XGBoost"]=XGBClassifier(n_estimators=500,learning_rate=0.05,max_depth=5,scale_pos_weight=w,random_state=SEED,verbosity=0,n_jobs=-1,eval_metric="logloss")
else:
    from sklearn.ensemble import AdaBoostClassifier
    top3["AdaBoost"]=AdaBoostClassifier(n_estimators=200,random_state=SEED)
top3["GradBoosting"]=GradientBoostingClassifier(n_estimators=300,learning_rate=0.05,max_depth=4,random_state=SEED)
print(f"Top 3: {list(top3.keys())}")

## 22. Final Training and Evaluation of Top 3

In [ ]:
X_tr_t=preprocessor.fit_transform(X_train); X_te_t=preprocessor.transform(X_test)
final={}
for name,mdl in top3.items():
    mdl.fit(X_tr_t,y_train); yp=mdl.predict(X_te_t); ypr=mdl.predict_proba(X_te_t)[:,1]
    final[name]={"Accuracy":accuracy_score(y_test,yp),"Precision":precision_score(y_test,yp),"Recall":recall_score(y_test,yp),"F1":f1_score(y_test,yp),"ROC-AUC":roc_auc_score(y_test,ypr)}
    print(f"\n{name}:"); print(classification_report(y_test,yp,target_names=["Not Potable","Potable"]))
pd.DataFrame(final).T

In [ ]:
fig,axes=plt.subplots(1,len(top3),figsize=(5*len(top3),4))
if len(top3)==1: axes=[axes]
for ax,(n,m) in zip(axes,top3.items()):
    ConfusionMatrixDisplay.from_predictions(y_test,m.predict(X_te_t),ax=ax,cmap="Blues",display_labels=["Not Potable","Potable"]); ax.set_title(n)
plt.tight_layout(); plt.show()

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(14,5))
for n,m in top3.items():
    RocCurveDisplay.from_estimator(m,X_te_t,y_test,ax=axes[0],name=n)
    PrecisionRecallDisplay.from_estimator(m,X_te_t,y_test,ax=axes[1],name=n)
axes[0].plot([0,1],[0,1],"k--",alpha=0.5)
plt.tight_layout(); plt.show()

## 23. Error Analysis

In [ ]:
bm=list(top3.values())[0]; ypb=bm.predict(X_te_t)
fn=(y_test==1)&(ypb==0); fp=(y_test==0)&(ypb==1)
print(f"FN: {fn.sum()}  FP: {fp.sum()}")
print(f"FN rate: {fn.sum()/(y_test==1).sum():.1%}")
print("\nNote: High error rate expected - features barely separate classes.")

## 24. Interpretation and Business Insight

**Key findings:** No single feature strongly predicts potability. Sulfate and pH have highest (but weak) correlations. Missing value patterns may carry information.

**Practical takeaway:** Real water testing relies on precise lab measurements, not ML from noisy features.

## 25. Limitations

1. Synthetic data
2. Weak signal
3. No spatial/temporal info
4. 14-24% missing values
5. Binary target (real potability is a spectrum)

## 26. How to Improve This Project

1. KNN imputation
2. Polynomial features
3. SVM with RBF kernel
4. Cross-validation (critical with small data)
5. Missingness indicators
6. Accept ~0.70 accuracy ceiling

## 27. Production Considerations

- Real water testing uses lab instruments, not ML
- ML could triage samples for priority testing
- Regulatory compliance requires exact WHO thresholds
- False negatives are life-threatening
- Model should be very conservative

## 28. Common Mistakes

1. Expecting high accuracy (65-70% is realistic)
2. Dropping rows with missing values (loses 25%+ data)
3. Not using class_weight
4. Over-engineering when signal is weak
5. Claiming production readiness

## 29. Mini Challenge / Exercises

1. Add missingness indicator features
2. Try KNN imputer
3. Polynomial features (degree=2)
4. What is the theoretical Bayes error rate?
5. If FN costs 100x FP, what threshold?

## 30. Final Summary / Key Takeaways

| Aspect | Detail |
|---|---|
| Task | Binary classification - water potability |
| Dataset | ~3,276 samples, 9 features, ~39% potable |
| Difficulty | Hard - weak correlations |
| Best metric | F1 |
| Baseline | ~61% |
| Realistic F1 | 0.55-0.70 |
| Key lesson | Not every dataset yields high accuracy |

**What you learned:**
- Handling significant missing values
- Fundamental accuracy limits
- Feature engineering from water chemistry
- Full benchmark -> AutoML -> top 3 pipeline